# 01 generate flights

Design decisions: PX4 main built for SITL and cached as a tarball in `data/px4_cache`; the `sensor_gps_sim` patch is idempotent and re-applied before every build; the toolchain is reinstalled on every fresh runtime; one fresh PX4 daemon per flight; a run is named by `RUN` and may be restricted to one family and subtype for top-ups; runs are resume-safe through `manifest.jsonl`.

In [ ]:
# ============================================================
# BOOTSTRAP  (top of every notebook in this project)
# ============================================================
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT   = "UAV_GNSS"
REPO_NAME = "uav-gnss-triage"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "MyDrive" / f"{PROJECT}_Research"
REPO_DIR    = DRIVE_ROOT / REPO_NAME

if not (DRIVE_MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

for dotfile in (".gitconfig", ".git-credentials"):
    src = DRIVE_ROOT / dotfile
    if src.exists():
        shutil.copy(src, Path.home() / dotfile)
cred = Path.home() / ".git-credentials"
if cred.exists():
    os.chmod(cred, 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    import paths as P
print("CWD:", os.getcwd(), "| credentials:", cred.exists())


In [ ]:
# ---- PX4 build tree: verified cache restore or source build, toolchain check, patch, rebuild, atomic re-cache ----
subprocess.run(["pip", "install", "-q", "mavsdk", "pyulog"], check=True)
cache = P.PX4_CACHE / "px4_autopilot.tgz"
P.PX4_CACHE.mkdir(parents=True, exist_ok=True)

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(r.stdout[-1500:], r.stderr[-1500:]); return r.returncode

if cache.exists() and sh(f"gzip -t {cache}") != 0:
    print("cache is corrupt, removing it"); cache.unlink()
if P.PX4_SRC.exists() and not (P.PX4_SRC / ".git" / "HEAD").exists():
    print("partial tree, removing it"); shutil.rmtree(P.PX4_SRC)
if not P.PX4_SRC.exists():
    if cache.exists():
        sh(f"tar -C /content -xzf {cache}")
    if not (P.PX4_SRC / ".git" / "HEAD").exists():
        if P.PX4_SRC.exists():
            shutil.rmtree(P.PX4_SRC)
        sh(f"git clone --recursive https://github.com/PX4/PX4-Autopilot.git {P.PX4_SRC}")
if shutil.which("ninja") is None:          # fresh runtime: toolchain and Python deps are not persisted
    sh("bash Tools/setup/ubuntu.sh --no-nuttx --no-sim-tools 2>&1 | tail -3", cwd=P.PX4_SRC)
sh(f"{sys.executable} {P.SRC / 'sih_spoof_patch.py'} {P.PX4_SRC}")
rc = sh("make px4_sitl_default > /tmp/px4_build.log 2>&1; rc=$?; tail -3 /tmp/px4_build.log; exit $rc", cwd=P.PX4_SRC)
assert rc == 0, "PX4 build failed"
tmp = cache.with_suffix(".tgz.tmp")
if sh(f"tar -C /content -czf {tmp} PX4-Autopilot") == 0 and sh(f"gzip -t {tmp}") == 0:
    os.replace(tmp, cache); print("cache refreshed")
else:
    print("cache NOT refreshed (tar or verify failed); build tree is still usable this session")
print("firmware:", subprocess.run(["git", "-C", str(P.PX4_SRC), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# ---- generate (resume-safe). Main run: RUN="sih_flights_v2", FAMILIES=None, ONLY_SUBTYPE=None, N=60, SEED=20260909, SCHEME="legacy".
# ---- Top-up: RUN="sih_flights_v2_nofix", FAMILIES="gps_degrade", ONLY_SUBTYPE="no_fix_then_ok", N=30, SEED=20260915, SCHEME="hashed".
RUN, FAMILIES, ONLY_SUBTYPE, N, SEED, SCHEME = "sih_flights_v2_nofix", "gps_degrade", "no_fix_then_ok", 30, 20260915, "hashed"
out = P.SIH_RUNS / RUN
args = [sys.executable, str(P.SRC / "sih_generate_flights.py"), "--px4", str(P.PX4_SRC), "--out", str(out),
        "--n_per_family", str(N), "--speed", "8", "--seed", str(SEED), "--seed_scheme", SCHEME]
if FAMILIES:
    args += ["--families", FAMILIES]
if ONLY_SUBTYPE:
    args += ["--only_subtype", ONLY_SUBTYPE]
proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    if not line.startswith("skip (done)"):
        print(line, end="")
print("exit code:", proc.wait())


In [ ]:
# ---- redo flights that failed in a previous run of RUN: drop their stubs, regenerate with identical seeds ----
import json
lines = [l for l in open(out / "manifest.jsonl") if l.strip()]
keep, drop = [], []
for l in lines:
    (keep if json.loads(l)["ok"] else drop).append(json.loads(l))
if drop:
    shutil.copy(out / "manifest.jsonl", out / "manifest_before_redo.jsonl")
    with open(out / "manifest.jsonl", "w") as f:
        for r in keep:
            f.write(json.dumps(r) + "\n")
    for r in drop:
        for ext in (".ulg", ".json", ".px4.txt"):
            p = out / f"{r['flight_id']}{ext}"
            if p.exists():
                p.unlink()
    print("kept", len(keep), "| dropped for regeneration", len(drop), "-> rerun the generation cell")
else:
    print("nothing to redo:", len(keep), "flights ok")


In [ ]:
# ---- manifest quality check for RUN ----
import json, pandas as pd
rows = [json.loads(l) for l in open(out / "manifest.jsonl") if l.strip()]
df = pd.json_normalize(rows)
print("flights:", len(df), "| ok:", int(df["ok"].sum()), "| with notes:", int((df["notes"].str.len() > 0).sum()))
print(df.groupby(["family", "subtype"]).size().to_string())
print(pd.Series([n.split(":")[0].split(" on attempt")[0] for ns in df["notes"] for n in ns]).value_counts().to_string())
print(df.groupby("family")["verify.max_gps_vs_truth_m"].describe()[["count", "min", "50%", "max"]].round(1).to_string())
print(df.groupby("family")["verify.duration_s"].describe()[["min", "50%", "max"]].round(0).to_string())
if "verify.onset_sim_s" in df:
    print("onset logged for", int(df["verify.onset_sim_s"].notna().sum()), "of", int((df.family != "nominal").sum()), "attacked flights")
if (df.family == "gps_degrade").any():
    print("gps_degrade last30s:", df[df.family == "gps_degrade"]["verify.div_last30s_max_m"].describe()[["50%", "max"]].round(1).to_dict())


In [ ]:
import json, collections
print(collections.Counter(json.loads(l)["firmware_commit"][:10] for l in open(out / "manifest.jsonl") if l.strip()))

In [ ]:
# ---- pin firmware to the v2 commit, rebuild, regenerate the no-fix top-up on it, QC ----
import json, collections
PIN = "e53ff6b3ffc38354973a9879ff87dda58c40c9e7"
cache = P.PX4_CACHE / "px4_autopilot.tgz"

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(r.stdout[-1200:], r.stderr[-1200:]); return r.returncode

head = subprocess.run(["git", "-C", str(P.PX4_SRC), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if head != PIN:
    sh("git checkout -q -- . && git submodule foreach -q --recursive 'git checkout -q -- .'", cwd=P.PX4_SRC)
    assert sh(f"git checkout -q {PIN} && git submodule update --init --recursive -q", cwd=P.PX4_SRC) == 0, "checkout failed"
    sh(f"{sys.executable} {P.SRC / 'sih_spoof_patch.py'} {P.PX4_SRC}")
    rc = sh("make px4_sitl_default > /tmp/px4_build.log 2>&1; rc=$?; tail -3 /tmp/px4_build.log; exit $rc", cwd=P.PX4_SRC)
    assert rc == 0, "PX4 build failed"
    tmp = cache.with_suffix(".tgz.tmp")
    if sh(f"tar -C /content -czf {tmp} PX4-Autopilot") == 0 and sh(f"gzip -t {tmp}") == 0:
        os.replace(tmp, cache); print("cache refreshed at pinned commit")
print("firmware now:", subprocess.run(["git", "-C", str(P.PX4_SRC), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

out = P.SIH_RUNS / "sih_flights_v2_nofix"
shutil.rmtree(out, ignore_errors=True)
proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_generate_flights.py"), "--px4", str(P.PX4_SRC), "--out", str(out),
                         "--families", "gps_degrade", "--only_subtype", "no_fix_then_ok", "--n_per_family", "30",
                         "--speed", "8", "--seed", "20260915", "--seed_scheme", "hashed"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    if line.startswith("{"):
        print(line, end="")
print("exit code:", proc.wait())

rows = [json.loads(l) for l in open(out / "manifest.jsonl") if l.strip()]
print("flights:", len(rows), "| ok:", sum(r["ok"] for r in rows), "| notes:", sum(bool(r["notes"]) for r in rows))
print("firmware:", collections.Counter(r["firmware_commit"][:10] for r in rows))